# Per Waterbody Statisitics

## Select waterbody

The annual water quality variables is a tile based workflow. 

Get the region code of a tile which has been processed from https://explorer.dev.digitalearth.africa/products/wq_annual.

In [1]:
tile_region_code = "x200y034"

In [2]:
from water_quality.tiling import parse_region_code
from water_quality.grid import get_waterbodies_grid

Get the spatial extent of the tile.

In [3]:
grid = get_waterbodies_grid()
tile_index = parse_region_code(tile_region_code)
tile_geobox = grid.tile_geobox(tile_index)
tile_geobox.explore()

Search for waterbodies within the tile extent from the Waterbodies Historical Extent product. 

In [4]:
from importlib.resources import files
from odc.geo.geobox import GeoBox
from odc.geo.geom import Geometry
import geopandas as gpd
from deafrica_tools.waterbodies import get_waterbodies

In [5]:
bbox = tuple(tile_geobox.extent.boundingbox.to_crs("EPSG:4326"))

In [6]:
# Filter the known test areas by the bounding box of the tile. 
place_name = "Lake_Baringo"
places_fp = files("water_quality.data").joinpath("places.parquet")
places_gdf = gpd.read_parquet(places_fp)

xmin, ymin, xmax, ymax = bbox
places = places_gdf.cx[xmin:xmax, ymin:ymax]
places

,name,description,geometry
23,Thewaterskool_SA,,"POLYGON ((19.3 -34.1, 19.3 -33.98, 19.1 -33.98..."
24,SA_dam,,"POLYGON ((19.47 -33.8, 19.47 -33.65, 19.35 -33..."
25,SA_dam_north,,"POLYGON ((19.44 -33.73, 19.44 -33.699, 19.42 -..."
26,SA_dam_south,,"POLYGON ((19.431 -33.781, 19.431 -33.772, 19.4..."
32,SA_smalldam,"Irrigation Dam, South Africa","POLYGON ((19.498 -33.802, 19.498 -33.8, 19.494..."
33,SA_smalldam1,"Irrigation Dam, South Africa, clear water","POLYGON ((19.51 -33.8065, 19.51 -33.803, 19.50..."


In [7]:
waterbodies = get_waterbodies(bbox)
waterbodies.head()

,id,wb_id,area_m2,length_m,uid,perim_m,last_obs_date,last_valid_obs_date,last_valid_obs,last_attrs_update_date,geometry
0,DEAfrica_Waterbodies.k3vvpnrmmn,90467,5400.0,90.000000,k3vvpnrmmn,300,2026-01-05,2026-01-05,0.000000,2026-01-13,"POLYGON ((19.6533 -34.2418, 19.6533 -34.2424, ..."
1,DEAfrica_Waterbodies.k3vwxd2ys6,90587,5400.0,120.000000,k3vwxd2ys6,360,2026-01-05,2026-01-05,16.666667,2026-01-13,"POLYGON ((19.3147 -33.9999, 19.3147 -34.0005, ..."
2,DEAfrica_Waterbodies.k3vthd9d2z,90291,6300.0,127.279566,k3vthd9d2z,360,2026-01-05,2025-12-19,85.714286,2026-01-13,"POLYGON ((19.1841 -34.2628, 19.1847 -34.2628, ..."
3,DEAfrica_Waterbodies.k3vthe670z,90292,6300.0,90.000000,k3vthe670z,360,2026-01-05,2026-01-05,0.000000,2026-01-13,"POLYGON ((19.185 -34.2585, 19.1856 -34.2585, 1..."
4,DEAfrica_Waterbodies.k3yne6khjn,91418,8100.0,170.763110,k3yne6khjn,480,2026-01-05,2026-01-05,0.000000,2026-01-13,"POLYGON ((19.8355 -33.9999, 19.8355 -34.0002, ..."


Click on the waterbody of interest in the map below and copy the waterbody geohash to use in the next step of this workflow. 

In [8]:
m = places.explore(color="red")
waterbodies.explore(m=m, popup="uid")

# Per waterbody summary

In [9]:
from deafrica_tools.waterbodies import get_waterbody
from odc.geo.geom import Geometry
from datacube import Datacube

Paste the geohash (uid) for the waterbody of interest in the cell below.

In [10]:
waterbody_uid = "k3vwmjdbmc"

In [11]:
waterbody = get_waterbody(waterbody_uid)
waterbody

,id,wb_id,area_m2,length_m,uid,perim_m,last_obs_date,last_valid_obs_date,last_valid_obs,last_attrs_update_date,geometry
0,DEAfrica_Waterbodies.k3vwmjdbmc,90561,4.764960e+07,17589.448221,k3vwmjdbmc,93180,2026-01-05,2025-12-19,83.371109,2026-01-13,"POLYGON ((19.1511 -33.9838, 19.1517 -33.9838, ..."


The continental workflow will only generate summaries for waterbodies with an area of 0.1 ha and above.

In [12]:
m2_per_ha = 10_000
# This threshold is too small.
area_threshold_ha = 0.1

area_m2_ha = waterbody["area_m2"] / m2_per_ha
generate_summary = bool((area_m2_ha > area_threshold_ha).all())

if generate_summary is False:
    waterbody = None
    print(f"Waterbody does not meet area threshod of {area_filter_ha} ha")

In [13]:
waterbody_geom = Geometry(waterbody["geometry"].iloc[0], waterbody.crs)

View all the measurements available for the annual product. 

In [14]:
dc = Datacube(app="WaterQuality")

dc.list_measurements().xs("wq_annual", level='product')

,name,dtype,units,nodata,aliases,flags_definition,add_offset,scale_factor
measurement,,,,,,,,
agm_fai,agm_fai,float32,1,NaN,NaN,NaN,0.0,1.0
agm_hue,agm_hue,float32,1,NaN,NaN,NaN,0.0,1.0
agm_ndvi,agm_ndvi,float32,1,NaN,NaN,NaN,0.0,1.0
agm_owt,agm_owt,float32,1,NaN,NaN,NaN,0.0,1.0
chla,chla,float32,1,NaN,NaN,NaN,0.0,1.0
clear_water,clear_water,float32,1,NaN,NaN,NaN,0.0,1.0
msi_agm_fai,msi_agm_fai,float32,1,NaN,NaN,NaN,0.0,1.0
msi_agm_hue,msi_agm_hue,float32,1,NaN,NaN,NaN,0.0,1.0
msi_agm_ndvi,msi_agm_ndvi,float32,1,NaN,NaN,NaN,0.0,1.0


In [15]:
# Use the historical extent polygon as the area of interest
# to load annual water quality measurements

measurements = ["agm_fai", "agm_ndvi", "agm_hue", "agm_owt", "tsi", "tsm", "tirs_st_ann_max", "tirs_st_ann_med", "tirs_st_ann_min", "water_mask"]

ds = dc.load(product="wq_annual", geopolygon=waterbody_geom, measurements=measurements, dask_chunks={"x": 300, "y":300})

ds

<xarray.Dataset> Size: 2GB
Dimensions:          (time: 25, y: 1016, x: 1680)
Coordinates:
  * time             (time) datetime64[ns] 200B 2000-07-01T23:59:59.999999 .....
  * y                (y) float64 8kB -4.091e+06 -4.091e+06 ... -4.102e+06
  * x                (x) float64 13kB 1.845e+06 1.845e+06 ... 1.862e+06
    spatial_ref      int32 4B 6933
Data variables:
    agm_fai          (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
    agm_ndvi         (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
    agm_hue          (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
    agm_owt          (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
    tsi              (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
    tsm              (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
    tirs_st_ann_max  (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
    tirs_st_ann_med  (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
    tirs_st_ann_min  (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
    water_mask       (time, y, x) float32 171MB dask.array<chunksize=(1, 300, 300), meta=np.ndarray>
Attributes:
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

In [16]:
assert ds.odc.geobox.crs.projected

In [17]:
resolution = abs(ds.odc.geobox.resolution.x)
resolution

10.0

In [18]:
m2_per_km2 = 1_000_000

In [19]:
# List to store the summary statisitcs tables
summary_tables = {}

In [20]:
ds["water_mask"].count(dim=('x','y'))

<xarray.DataArray 'water_mask' (time: 25)> Size: 200B
dask.array<sum-aggregate, shape=(25,), dtype=int64, chunksize=(1,), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 200B 2000-07-01T23:59:59.999999 ... 20...
    spatial_ref  int32 4B 6933
Attributes:
    units:         1
    nodata:        nan
    add_offset:    0.0
    scale_factor:  1.0
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

In [21]:
# Area of water detected for the waterbody each year. 
water_area_km2 = ((ds["water_mask"].count(dim=('x','y')) * resolution) / m2_per_km2).compute()
# water_area_km2 = water_area_km2.to_dataframe().drop(columns=["spatial_ref"])

summary_tables["water_area"] = water_area_km2.to_dataframe(name="water_area").drop(columns=["spatial_ref"])

water_area_km2

<xarray.DataArray 'water_mask' (time: 25)> Size: 200B
array([4.63446, 4.64688, 4.66767, 4.4901 , 4.20057, 4.06341, 4.07268,
       3.75831, 3.81204, 4.47165, 4.55994, 4.62384, 4.65786, 4.6071 ,
       4.75767, 4.57353, 4.41405, 4.2171 , 3.64167, 3.32091, 3.25494,
       3.57867, 4.30614, 4.46994, 4.60107])
Coordinates:
  * time         (time) datetime64[ns] 200B 2000-07-01T23:59:59.999999 ... 20...
    spatial_ref  int32 4B 6933
Attributes:
    units:         1
    nodata:        nan
    add_offset:    0.0
    scale_factor:  1.0
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

In [22]:
# Area of water with consistent algae during the year 
fai_area_km2 = ((ds["agm_fai"].count(dim=('x','y')) * resolution) / m2_per_km2).compute()
fai_area_km2

<xarray.DataArray 'agm_fai' (time: 25)> Size: 200B
array([0.15461, 0.14503, 0.01248, 0.07713, 0.26906, 0.10817, 0.01641,
       0.00976, 0.00504, 0.00656, 0.01007, 0.1135 , 0.12365, 0.01509,
       0.00752, 0.09063, 0.50934, 0.56504, 0.1732 , 0.00343, 0.00209,
       0.00171, 0.00293, 0.03301, 0.00942])
Coordinates:
  * time         (time) datetime64[ns] 200B 2000-07-01T23:59:59.999999 ... 20...
    spatial_ref  int32 4B 6933
Attributes:
    units:         1
    nodata:        nan
    add_offset:    0.0
    scale_factor:  1.0
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

In [23]:
# Area of water with consistent vegetation during the year 
ndvi_area_km2 = ((ds["agm_ndvi"].count(dim=('x','y')) * resolution) / m2_per_km2).compute()
ndvi_area_km2

<xarray.DataArray 'agm_ndvi' (time: 25)> Size: 200B
array([0.4435 , 0.73294, 0.1442 , 0.33025, 1.06843, 0.53112, 0.1776 ,
       0.11517, 0.04166, 0.07883, 0.10468, 0.39028, 0.51466, 0.09898,
       0.07041, 0.33782, 1.14698, 2.1154 , 0.58888, 0.04242, 0.0185 ,
       0.01752, 0.03384, 0.0969 , 0.04849])
Coordinates:
  * time         (time) datetime64[ns] 200B 2000-07-01T23:59:59.999999 ... 20...
    spatial_ref  int32 4B 6933
Attributes:
    units:         1
    nodata:        nan
    add_offset:    0.0
    scale_factor:  1.0
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

In [24]:
# Percent of the water area consistently indicating algae during the year
fai_cover = (fai_area_km2 / water_area_km2).to_dataframe(name="fai_cover_percent").drop(columns=["spatial_ref"])
summary_tables["fai_cover"] = fai_cover
fai_cover

,fai_cover_percent
time,
2000-07-01 23:59:59.999999,0.033361
2001-07-02 11:59:59.999999,0.031210
2002-07-02 11:59:59.999999,0.002674
2003-07-02 11:59:59.999999,0.017178
2004-07-01 23:59:59.999999,0.064053
2005-07-02 11:59:59.999999,0.026620
2006-07-02 11:59:59.999999,0.004029
2007-07-02 11:59:59.999999,0.002597
2008-07-01 23:59:59.999999,0.001322


In [25]:
# Percent of the water area consistently indicating vegetation during the year
ndvi_cover = (ndvi_area_km2 / water_area_km2).to_dataframe(name="ndvi_cover_percent").drop(columns=["spatial_ref"])
summary_tables["ndvi_cover"] = ndvi_cover
ndvi_cover

,ndvi_cover_percent
time,
2000-07-01 23:59:59.999999,0.095696
2001-07-02 11:59:59.999999,0.157727
2002-07-02 11:59:59.999999,0.030893
2003-07-02 11:59:59.999999,0.073551
2004-07-01 23:59:59.999999,0.254354
2005-07-02 11:59:59.999999,0.130708
2006-07-02 11:59:59.999999,0.043608
2007-07-02 11:59:59.999999,0.030644
2008-07-01 23:59:59.999999,0.010929


In [26]:
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

In [27]:
import pandas as pd
def flatten_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.unstack("quantile")
    df.columns = [f"{col}_q{val}" for col, val in df.columns]
    return df

In [28]:
# Annual quantiles for hue. 
hue_quantiles = ds["agm_hue"].quantile(quantiles, dim=('x','y')).compute()
hue_summary = flatten_df(hue_quantiles.to_dataframe(name="hue"))
summary_tables["hue_summary"] = hue_summary
hue_summary

,hue_q0.1,hue_q0.2,hue_q0.3,hue_q0.4,hue_q0.5,hue_q0.6,hue_q0.7,hue_q0.8,hue_q0.9
time,,,,,,,,,
2000-07-01 23:59:59.999999,48.999935,51.476350,52.572077,53.382346,54.106544,54.858876,55.732062,56.946928,59.238510
2001-07-02 11:59:59.999999,39.887891,43.298931,45.048027,45.934643,46.567974,47.075493,47.543915,48.051247,48.744821
2002-07-02 11:59:59.999999,44.968872,46.247762,46.775898,47.121262,47.402077,47.670708,47.965729,48.375891,49.186905
2003-07-02 11:59:59.999999,40.724570,42.329788,43.072488,43.693932,44.272911,44.866833,45.506336,46.216859,47.260536
2004-07-01 23:59:59.999999,38.864084,40.155094,40.867538,41.815731,43.092678,43.683556,44.048943,44.366024,44.746197
2005-07-02 11:59:59.999999,36.653317,38.321695,39.338931,40.185885,40.919025,41.461969,41.779499,42.030457,42.334123
2006-07-02 11:59:59.999999,38.817318,40.201294,40.596718,40.825188,41.013273,41.203384,41.428360,41.761032,42.387785
2007-07-02 11:59:59.999999,39.592556,41.365860,41.982448,42.293643,42.527901,42.737164,42.949585,43.192981,43.544720
2008-07-01 23:59:59.999999,43.072687,43.633930,44.025051,44.355991,44.668234,45.003815,45.402752,45.953999,46.934139


In [29]:
# Annual quantiles for optical water type. 
owt_quantiles = ds["agm_owt"].quantile(quantiles, dim=('x','y')).compute()
owt_summary = flatten_df(owt_quantiles.to_dataframe(name="owt"))
summary_tables["owt_summary"] = owt_summary
owt_summary

,owt_q0.1,owt_q0.2,owt_q0.3,owt_q0.4,owt_q0.5,owt_q0.6,owt_q0.7,owt_q0.8,owt_q0.9
time,,,,,,,,,
2000-07-01 23:59:59.999999,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2001-07-02 11:59:59.999999,1.0,1.0,1.0,1.0,1.0,1.0,5.0,6.0,6.0
2002-07-02 11:59:59.999999,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,5.0
2003-07-02 11:59:59.999999,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0,6.0
2004-07-01 23:59:59.999999,1.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,6.0
2005-07-02 11:59:59.999999,1.0,1.0,1.0,1.0,6.0,6.0,6.0,6.0,7.0
2006-07-02 11:59:59.999999,1.0,1.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0
2007-07-02 11:59:59.999999,1.0,1.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0
2008-07-01 23:59:59.999999,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [30]:
# Annual quantiles for Trophic State
tsi_quantiles = ds["tsi"].quantile(quantiles, dim=('x','y')).compute()
tsi_summary = flatten_df(tsi_quantiles.to_dataframe(name="tsi"))
summary_tables["tsi_summary"] = tsi_summary
tsi_summary

,tsi_q0.1,tsi_q0.2,tsi_q0.3,tsi_q0.4,tsi_q0.5,tsi_q0.6,tsi_q0.7,tsi_q0.8,tsi_q0.9
time,,,,,,,,,
2000-07-01 23:59:59.999999,45.717106,53.582546,57.654314,61.263779,63.304817,64.674248,65.733498,67.339180,74.172081
2001-07-02 11:59:59.999999,0.000000,0.000000,0.000000,0.000000,42.226709,53.171791,61.531257,66.347777,75.503551
2002-07-02 11:59:59.999999,0.000000,0.000000,0.000000,0.000000,28.201197,44.537064,50.770330,56.669131,65.711511
2003-07-02 11:59:59.999999,0.000000,0.000000,0.000000,0.000000,37.856043,48.017780,53.007992,59.185848,68.167293
2004-07-01 23:59:59.999999,0.000000,0.000000,0.000000,0.000000,55.833981,62.968056,64.602991,72.750229,77.317459
2005-07-02 11:59:59.999999,40.464844,47.615120,51.724327,56.470863,62.696590,64.133224,65.274567,68.192558,76.088631
2006-07-02 11:59:59.999999,48.587490,50.974157,52.976094,55.429303,60.241331,61.888684,62.518702,63.499997,68.167014
2007-07-02 11:59:59.999999,0.000000,0.000000,27.837826,43.807777,53.210873,59.550560,61.517136,62.415981,65.320076
2008-07-01 23:59:59.999999,0.000000,0.000000,0.000000,34.391506,45.972506,51.331359,54.464932,56.868633,60.016870


In [31]:
# Annual quantiles Turbidity
turbidity_quantiles = ds["tsm"].quantile(quantiles, dim=('x','y')).compute()
turbidity_summary = flatten_df(tsi_quantiles.to_dataframe(name="tsm"))
summary_tables["turbidity_summary"] = turbidity_summary

turbidity_summary

,tsm_q0.1,tsm_q0.2,tsm_q0.3,tsm_q0.4,tsm_q0.5,tsm_q0.6,tsm_q0.7,tsm_q0.8,tsm_q0.9
time,,,,,,,,,
2000-07-01 23:59:59.999999,45.717106,53.582546,57.654314,61.263779,63.304817,64.674248,65.733498,67.339180,74.172081
2001-07-02 11:59:59.999999,0.000000,0.000000,0.000000,0.000000,42.226709,53.171791,61.531257,66.347777,75.503551
2002-07-02 11:59:59.999999,0.000000,0.000000,0.000000,0.000000,28.201197,44.537064,50.770330,56.669131,65.711511
2003-07-02 11:59:59.999999,0.000000,0.000000,0.000000,0.000000,37.856043,48.017780,53.007992,59.185848,68.167293
2004-07-01 23:59:59.999999,0.000000,0.000000,0.000000,0.000000,55.833981,62.968056,64.602991,72.750229,77.317459
2005-07-02 11:59:59.999999,40.464844,47.615120,51.724327,56.470863,62.696590,64.133224,65.274567,68.192558,76.088631
2006-07-02 11:59:59.999999,48.587490,50.974157,52.976094,55.429303,60.241331,61.888684,62.518702,63.499997,68.167014
2007-07-02 11:59:59.999999,0.000000,0.000000,27.837826,43.807777,53.210873,59.550560,61.517136,62.415981,65.320076
2008-07-01 23:59:59.999999,0.000000,0.000000,0.000000,34.391506,45.972506,51.331359,54.464932,56.868633,60.016870


In [32]:
min_temp = flatten_df(ds["tirs_st_ann_min"].quantile(quantiles, dim=('x','y')).compute().to_dataframe(name="min_temp"))
median_temp = flatten_df(ds["tirs_st_ann_med"].quantile(quantiles, dim=('x','y')).compute().to_dataframe(name="median_temp"))
max_temp = flatten_df(ds["tirs_st_ann_max"].quantile(quantiles, dim=('x','y')).compute().to_dataframe(name="max_temp"))

temp_summary = min_temp.join(median_temp).join(max_temp)
summary_tables["temp_summary"] = temp_summary
temp_summary

,min_temp_q0.1,min_temp_q0.2,min_temp_q0.3,min_temp_q0.4,min_temp_q0.5,min_temp_q0.6,min_temp_q0.7,min_temp_q0.8,min_temp_q0.9,median_temp_q0.1,...,median_temp_q0.9,max_temp_q0.1,max_temp_q0.2,max_temp_q0.3,max_temp_q0.4,max_temp_q0.5,max_temp_q0.6,max_temp_q0.7,max_temp_q0.8,max_temp_q0.9
time,,,,,,,,,,,,,,,,,,,,,
2000-07-01 23:59:59.999999,10.178972,10.932898,11.343738,11.744309,12.105957,12.576649,13.377658,14.053090,14.581509,17.971283,...,23.203115,24.180147,25.620450,26.613279,27.237518,27.832256,28.422205,29.172150,30.148655,31.729923
2001-07-02 11:59:59.999999,12.391471,12.547560,12.606277,12.684733,12.771337,12.935790,13.150513,13.483368,13.944855,19.904721,...,24.290059,22.848558,23.277550,23.580231,23.953566,24.319566,24.672979,25.694035,27.888765,30.883295
2002-07-02 11:59:59.999999,10.888902,11.084182,11.259029,11.408092,11.531708,11.691070,11.853027,12.059468,12.321973,17.263763,...,20.179321,23.062523,23.279844,23.417307,23.549469,23.683226,23.781891,23.915192,24.146494,24.952557
2003-07-02 11:59:59.999999,16.777243,17.438520,17.673920,17.868971,18.154030,18.560312,19.150786,20.018396,21.279250,22.079742,...,26.526410,24.859134,25.121796,25.297251,25.491089,25.659176,25.887421,26.179998,26.889607,29.347794
2004-07-01 23:59:59.999999,11.617157,11.983584,12.194369,12.458689,12.633388,12.963898,13.518741,14.330401,15.145374,15.571808,...,24.697937,22.670898,23.126415,23.732346,24.410756,25.105377,25.784882,27.119257,29.392597,32.683120
2005-07-02 11:59:59.999999,11.463576,11.924906,12.226954,12.377194,12.507340,12.630618,12.829804,13.059158,13.517609,17.638981,...,21.626846,23.119219,23.453880,23.842397,24.718445,25.471292,27.212362,30.023286,32.981532,35.649498
2006-07-02 11:59:59.999999,11.304761,11.528290,11.669474,11.794581,11.894037,11.999316,12.103679,12.252869,12.483984,14.546039,...,20.118925,23.974550,24.480781,24.821003,25.098541,25.420599,25.718795,26.038038,26.539471,28.128271
2007-07-02 11:59:59.999999,10.444794,10.693052,11.014835,11.233281,11.384735,11.468589,11.588235,11.754753,12.088182,16.398977,...,20.030060,22.230324,22.506533,22.648949,22.803213,22.993023,23.169847,23.290121,23.508226,24.123889
2008-07-01 23:59:59.999999,11.496180,11.738325,11.946164,12.122206,12.333821,12.543457,12.750150,13.041371,13.485297,18.928314,...,21.888336,23.978616,24.292032,24.503872,24.699647,24.895287,25.111908,25.368116,25.648272,26.059345


In [33]:
import pandas as pd
per_waterbody_summary = pd.concat(list(summary_tables.values()), axis=1)
per_waterbody_summary

,water_area,fai_cover_percent,ndvi_cover_percent,hue_q0.1,hue_q0.2,hue_q0.3,hue_q0.4,hue_q0.5,hue_q0.6,hue_q0.7,...,median_temp_q0.9,max_temp_q0.1,max_temp_q0.2,max_temp_q0.3,max_temp_q0.4,max_temp_q0.5,max_temp_q0.6,max_temp_q0.7,max_temp_q0.8,max_temp_q0.9
time,,,,,,,,,,,,,,,,,,,,,
2000-07-01 23:59:59.999999,4.63446,0.033361,0.095696,48.999935,51.476350,52.572077,53.382346,54.106544,54.858876,55.732062,...,23.203115,24.180147,25.620450,26.613279,27.237518,27.832256,28.422205,29.172150,30.148655,31.729923
2001-07-02 11:59:59.999999,4.64688,0.031210,0.157727,39.887891,43.298931,45.048027,45.934643,46.567974,47.075493,47.543915,...,24.290059,22.848558,23.277550,23.580231,23.953566,24.319566,24.672979,25.694035,27.888765,30.883295
2002-07-02 11:59:59.999999,4.66767,0.002674,0.030893,44.968872,46.247762,46.775898,47.121262,47.402077,47.670708,47.965729,...,20.179321,23.062523,23.279844,23.417307,23.549469,23.683226,23.781891,23.915192,24.146494,24.952557
2003-07-02 11:59:59.999999,4.49010,0.017178,0.073551,40.724570,42.329788,43.072488,43.693932,44.272911,44.866833,45.506336,...,26.526410,24.859134,25.121796,25.297251,25.491089,25.659176,25.887421,26.179998,26.889607,29.347794
2004-07-01 23:59:59.999999,4.20057,0.064053,0.254354,38.864084,40.155094,40.867538,41.815731,43.092678,43.683556,44.048943,...,24.697937,22.670898,23.126415,23.732346,24.410756,25.105377,25.784882,27.119257,29.392597,32.683120
2005-07-02 11:59:59.999999,4.06341,0.026620,0.130708,36.653317,38.321695,39.338931,40.185885,40.919025,41.461969,41.779499,...,21.626846,23.119219,23.453880,23.842397,24.718445,25.471292,27.212362,30.023286,32.981532,35.649498
2006-07-02 11:59:59.999999,4.07268,0.004029,0.043608,38.817318,40.201294,40.596718,40.825188,41.013273,41.203384,41.428360,...,20.118925,23.974550,24.480781,24.821003,25.098541,25.420599,25.718795,26.038038,26.539471,28.128271
2007-07-02 11:59:59.999999,3.75831,0.002597,0.030644,39.592556,41.365860,41.982448,42.293643,42.527901,42.737164,42.949585,...,20.030060,22.230324,22.506533,22.648949,22.803213,22.993023,23.169847,23.290121,23.508226,24.123889
2008-07-01 23:59:59.999999,3.81204,0.001322,0.010929,43.072687,43.633930,44.025051,44.355991,44.668234,45.003815,45.402752,...,21.888336,23.978616,24.292032,24.503872,24.699647,24.895287,25.111908,25.368116,25.648272,26.059345
